# 00-01 — Operational GloFAS Forecast Downloader

Downloads daily operational GloFAS forecasts for named flood events.
Each forecast initialisation date is requested individually; each request
covers **15 days of lead time** (24 h … 360 h, step 24 h).

**Window per event**: 15 days before peak → 3 days after peak (19 days total).

**Pattern**: submit ALL requests first (Phase 1), then poll + download as
each becomes ready (Phase 2). State is persisted to `jobs_state.json` so
the notebook can be safely interrupted and re-run.

**Only Cell 2 needs editing** to add events or change parameters.

In [1]:
# Cell 1 — Imports
import os
import json
import time
import random
import zipfile
import shutil
from pathlib import Path
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor
from collections import Counter

import requests
from tqdm.auto import tqdm
from ecmwf.datastores import Client as DSClient

In [4]:
# Cell 2 — Configuration  ← only cell analysts need to edit

# ── Output root ───────────────────────────────────────────────────────────────
# This path (RAW_ROOT) is used only for output; it is the directory where
# downloaded forecast files will be saved. It is not read as input elsewhere
# in this notebook. If the folder is empty, that is expected for a new run.

# For cross-platform compatibility, use Path.home() and forward slashes
RAW_ROOT = Path.home() / "My Drive" / "GLOFAS_ImpactFloodForecasting_PHL" / "data" / "raw" / "glofas" / "forecast" / "Operational"

# Windows users may prefer to use their mapped Google Drive letter (uncomment and adjust path as needed)
# RAW_ROOT = Path(
#     r"G:\My Drive\GLOFAS_ImpactFloodForecasting_PHL"
#     r"\data\raw\glofas\forecast\Operational"
# )

# ── EWDS / API ────────────────────────────────────────────────────────────────
DATASET         = "cems-glofas-forecast"
SYSTEM_VERSION  = ["operational"]
PRODUCT_TYPE    = ["ensemble_perturbed_forecasts"]
VARIABLE        = "river_discharge_in_the_last_24_hours"
DATA_FORMAT     = "grib2"
DOWNLOAD_FORMAT = "zip"
AREA            = [35, 63, 4, 131]          # [N, W, S, E]

# Lead times: 24 h … 360 h, step 24 h  (15 days ahead)
LEADTIMES = [str(h) for h in range(24, 361, 24)]

# ── Events ────────────────────────────────────────────────────────────────────
EVENTS = {
    "EV_MARCE2024": {
        "label":     "TY Marce — Nov 2024",
        "peak_date": "2024-11-08",
    },
    "EV_UWAN2025": {
        "label":     "STY Uwan — Nov 2025",
        "peak_date": "2025-11-11",   # EDIT: confirm date
    },
    "EV_NEM2025": {
        "label":     "NE Monsoon — Dec 2025",
        "peak_date": "2025-11-29",   # EDIT: confirm date
    },
}

WINDOW_PRE_DAYS  = 15   # days before peak  (inclusive)
WINDOW_POST_DAYS =  3   # days after peak   (inclusive)

# ── Resume / disk controls ────────────────────────────────────────────────────
FORCE            = False   # True → re-download even if data.grib exists
KEEP_ZIPS        = False   # delete staging ZIPs after extraction
DEDUP_OVERLAP    = True    # one API job for days shared by two events; copy to both dirs
DAY_OUT_NAME     = "data.grib"

# ── Concurrency / polling ─────────────────────────────────────────────────────
MAX_INFLIGHT        = 12
DOWNLOAD_WORKERS    =  4
POLL_SECONDS        = 30
JITTER_SECONDS      =  5
MAX_SUBMIT_RETRIES  =  3
MAX_DOWNLOAD_RETRIES=  3

# ── Credentials ───────────────────────────────────────────────────────────────
# Expects .cdsapirc in the same directory as this notebook (same as NB00)
rc_path = Path(".") / ".cdsapirc"
assert rc_path.exists(), f".cdsapirc not found: {rc_path.resolve()}"
os.environ["CDSAPI_RC"]  = str(rc_path.resolve())
os.environ["CDSAPI_URL"] = "https://ewds.climate.copernicus.eu/api"

In [5]:
# Cell 3 — IO helpers
# ── Verbatim from 00_download_ECMWF.ipynb ────────────────────────────────────

def parse_cdsapirc(path: Path) -> tuple[str, str]:
    url = key = None
    for line in path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if s.startswith("url:"):
            url = s.split(":", 1)[1].strip()
        if s.startswith("key:"):
            key = s.split(":", 1)[1].strip()
    if not url or not key:
        raise RuntimeError(f"Could not parse url/key from {path}")
    token = key.split(":", 1)[1] if ":" in key else key
    return url, token

def tlog(msg: str) -> None:
    try:
        tqdm.write(msg)
    except Exception:
        print(msg, flush=True)

def mb(p: Path) -> float:
    return p.stat().st_size / (1024 * 1024)

def safe_unlink(path: Path) -> None:
    try:
        if path.exists():
            path.unlink()
    except Exception:
        pass

def safe_rmtree(path: Path) -> None:
    try:
        if path.exists():
            shutil.rmtree(path, ignore_errors=True)
    except Exception:
        pass

def is_valid_zip(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    if not zipfile.is_zipfile(path):
        return False
    try:
        with zipfile.ZipFile(path, "r") as zf:
            _ = zf.namelist()[:5]
        return True
    except Exception:
        return False

def is_grib_payload(p: Path) -> bool:
    """True if p is a non-empty GRIB file (starts with magic bytes b'GRIB')."""
    if (not p.is_file()) or p.stat().st_size == 0:
        return False
    if p.name.lower().endswith(".idx"):
        return False
    try:
        with open(p, "rb") as f:
            return f.read(4) == b"GRIB"
    except Exception:
        return False

def is_job_not_found_error(e: Exception) -> bool:
    if isinstance(e, requests.HTTPError):
        resp = getattr(e, "response", None)
        if resp is not None and getattr(resp, "status_code", None) == 404:
            return True
    msg = str(e).lower()
    return ("404" in msg) and ("job not found" in msg or "deleted" in msg)

def is_bad_request_400(e: Exception) -> bool:
    msg = str(e).lower()
    if "400" in msg and ("bad request" in msg or "invalid request" in msg):
        return True
    if isinstance(e, requests.HTTPError):
        resp = getattr(e, "response", None)
        if resp is not None and getattr(resp, "status_code", None) == 400:
            return True
    return False

# ── Day-level IO helpers (adapted from month-level equivalents) ───────────────

def has_day_data_grib(day_dir: Path) -> bool:
    """True if day_dir/data.grib is a valid GRIB payload."""
    return is_grib_payload(day_dir / DAY_OUT_NAME)

def normalize_day_folder(day_dir: Path) -> Path:
    """
    Ensure day_dir contains exactly one GRIB payload named data.grib.
    One request per day yields one GRIB, so no concatenation is expected.
    If multiple payloads appear (unexpected), concatenate with a warning.
    """
    day_dir.mkdir(parents=True, exist_ok=True)
    target = day_dir / DAY_OUT_NAME

    if is_grib_payload(target):
        for idx in day_dir.rglob("*.idx"):
            safe_unlink(idx)
        return target

    payloads = [p for p in day_dir.rglob("*.grib*") if is_grib_payload(p)]
    if not payloads:
        raise RuntimeError(f"No GRIB payloads found in: {day_dir}")

    if len(payloads) == 1:
        src = payloads[0]
        if src.resolve() != target.resolve():
            if target.exists():
                safe_unlink(target)
            src.replace(target)
        for idx in day_dir.rglob("*.idx"):
            safe_unlink(idx)
        return target

    # Multiple payloads (unexpected) — concatenate in sorted order
    tlog(f"WARNING: {day_dir.name} has {len(payloads)} GRIB payloads; concatenating")
    payloads = sorted(payloads)
    tmp = day_dir / (DAY_OUT_NAME + ".tmp")
    buf = 64 * 1024 * 1024
    with open(tmp, "wb") as w:
        for part in payloads:
            with open(part, "rb") as r:
                shutil.copyfileobj(r, w, length=buf)
    if target.exists():
        safe_unlink(target)
    tmp.replace(target)
    for part in payloads:
        if part.exists() and part.resolve() != target.resolve():
            safe_unlink(part)
    for idx in day_dir.rglob("*.idx"):
        safe_unlink(idx)
    if not is_grib_payload(target):
        raise RuntimeError(f"normalize_day_folder produced invalid data.grib: {target}")
    return target

def _extract_and_normalize(t: dict) -> None:
    """
    Extract ZIP → normalize day_dir to data.grib → copy to extra_day_dirs.
    Deletes ZIP unless KEEP_ZIPS is True.
    """
    zip_path = t["zip_path"]
    day_dir  = t["day_dir"]
    day_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(day_dir)
    if not KEEP_ZIPS:
        safe_unlink(zip_path)

    normalize_day_folder(day_dir)

    # Copy (not move) to additional event dirs for deduplicated days
    for extra_dir in t.get("extra_day_dirs", []):
        extra_dir.mkdir(parents=True, exist_ok=True)
        dest = extra_dir / DAY_OUT_NAME
        if not is_grib_payload(dest):
            shutil.copy2(day_dir / DAY_OUT_NAME, dest)
            tlog(f"  Copied → {extra_dir.parent.name}/{extra_dir.name}/{DAY_OUT_NAME}")

# ── Task builder ──────────────────────────────────────────────────────────────

def build_task_list(events: dict, dedup: bool = True) -> list[dict]:
    """
    Expand EVENTS into a flat list of per-day task dicts.

    Each task has:
      tag, event_id, day_str, year, month, day,
      day_dir, zip_path, extra_day_dirs, request

    If dedup=True, days shared between events produce a single task
    with extra_day_dirs populated; the worker copies data.grib to all dirs.
    """
    seen: dict[str, dict] = {}   # dedup_key → task
    tasks: list[dict] = []

    for event_id, ev in events.items():
        peak    = date.fromisoformat(ev["peak_date"])
        start_d = peak - timedelta(days=WINDOW_PRE_DAYS)
        end_d   = peak + timedelta(days=WINDOW_POST_DAYS)

        d = start_d
        while d <= end_d:
            day_str  = d.isoformat()       # "YYYY-MM-DD"
            year_s   = str(d.year)
            month_s  = f"{d.month:02d}"
            day_s    = f"{d.day:02d}"

            event_dir = RAW_ROOT / event_id
            day_dir   = event_dir / day_str
            staging   = event_dir / "_staging"
            zip_path  = staging / "zips" / f"{event_id}_{day_str}.zip"

            # Same calendar day → identical request body; deduplicate if requested
            dedup_key = day_str

            if dedup and dedup_key in seen:
                seen[dedup_key]["extra_day_dirs"].append(day_dir)
            else:
                tag = f"{event_id}_{day_str}"
                task = {
                    "tag":            tag,
                    "event_id":       event_id,
                    "day_str":        day_str,
                    "year":           year_s,
                    "month":          month_s,
                    "day":            day_s,
                    "day_dir":        day_dir,
                    "extra_day_dirs": [],
                    "zip_path":       zip_path,
                    "request": {
                        "system_version":    SYSTEM_VERSION,
                        "hydrological_model": ["lisflood"],
                        "product_type":      PRODUCT_TYPE,
                        "variable":          VARIABLE,
                        "year":              year_s,
                        "month":             month_s,
                        "day":               day_s,
                        "leadtime_hour":     LEADTIMES,
                        "data_format":       DATA_FORMAT,
                        "download_format":   DOWNLOAD_FORMAT,
                        "area":              AREA,
                    },
                }
                tasks.append(task)
                if dedup:
                    seen[dedup_key] = task

            d += timedelta(days=1)

    return tasks

In [4]:
# Cell 4 — State management

GLOBAL_STATE_PATH = RAW_ROOT / "_staging" / "jobs_state.json"

def save_state(state: dict) -> None:
    """Atomic write via temp rename — safe if kernel crashes mid-write."""
    GLOBAL_STATE_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = GLOBAL_STATE_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=2), encoding="utf-8")
    tmp.replace(GLOBAL_STATE_PATH)

def load_state(tasks: list[dict]) -> dict:
    """
    Load (or initialise) state, then reconcile each task against disk:
    - day_dir/data.grib exists   → mark done
    - state says done but file missing → reset to pending
    - stale valid ZIP but no data.grib → extract now
    - mid-download crash (no request_id) → reset to pending
    """
    if GLOBAL_STATE_PATH.exists():
        state = json.loads(GLOBAL_STATE_PATH.read_text(encoding="utf-8"))
    else:
        state = {"tasks": {}}

    for t in tasks:
        tag = t["tag"]
        if tag not in state["tasks"]:
            state["tasks"][tag] = {
                "status":      "pending",
                "request_id":  None,
                "attempts":    0,
                "last_error":  None,
            }

        rec = state["tasks"][tag]

        if not FORCE and has_day_data_grib(t["day_dir"]):
            rec["status"] = "done"
        elif rec.get("status") == "done" and not has_day_data_grib(t["day_dir"]):
            tlog(f"WARNING: {tag} marked done but data.grib missing → resetting")
            rec["status"] = "pending"
            rec["request_id"] = None
        elif rec.get("status") in ("downloading", "ready") and not rec.get("request_id"):
            rec["status"] = "pending"

        # Recover from a stale valid ZIP left behind by a previous crash
        if rec["status"] != "done" and is_valid_zip(t["zip_path"]):
            try:
                _extract_and_normalize(t)
                rec["status"] = "done"
                tlog(f"  Recovered {tag} from stale zip")
            except Exception as e:
                safe_unlink(t["zip_path"])
                tlog(f"  ZIP recovery failed for {tag}: {e}")

    save_state(state)
    return state

In [5]:
# Cell 5 — Startup: build task list, create dirs, load state

EWDS_URL, EWDS_TOKEN = parse_cdsapirc(rc_path)
ds = DSClient(url=EWDS_URL, key=EWDS_TOKEN)

# Expand events → flat task list
tasks = build_task_list(EVENTS, dedup=DEDUP_OVERLAP)

# Create staging directories for all events up front
RAW_ROOT.mkdir(parents=True, exist_ok=True)
(RAW_ROOT / "_staging").mkdir(parents=True, exist_ok=True)
for event_id in EVENTS:
    (RAW_ROOT / event_id / "_staging" / "zips").mkdir(parents=True, exist_ok=True)

# Load / initialise state, reconcile against disk
state = load_state(tasks)

# Summary
total    = len(tasks)
n_done   = sum(1 for t in tasks if state["tasks"][t["tag"]]["status"] == "done")
n_remain = total - n_done
print(f"Tasks : {total} total | {n_done} already done | {n_remain} to download")
print(f"Events: {list(EVENTS.keys())}")
if DEDUP_OVERLAP:
    shared = [t["tag"] for t in tasks if t["extra_day_dirs"]]
    if shared:
        print(f"Deduplicated days (1 API request → copies to both event dirs):")
        for tag in shared:
            t = next(x for x in tasks if x["tag"] == tag)
            dirs = [t["day_dir"]] + t["extra_day_dirs"]
            print(f"  {tag}  →  {[str(d.parent.name + '/' + d.name) for d in dirs]}")

Tasks : 56 total | 0 already done | 56 to download
Events: ['EV_MARCE2024', 'EV_UWAN2025', 'EV_NEM2025']
Deduplicated days (1 API request → copies to both event dirs):
  EV_UWAN2025_2025-11-14  →  ['EV_UWAN2025/2025-11-14', 'EV_NEM2025/2025-11-14']


In [6]:
# Cell 6 — Main execution
# Phase 1 (submit) and Phase 2 (poll + download) interleave in a single loop.

# ── Worker functions ──────────────────────────────────────────────────────────

def download_day_worker(tag: str, request_id: str, t: dict) -> bool:
    """
    Thread worker: download ZIP → extract → normalize → copy to extra_day_dirs.
    Uses a thread-local DSClient instance to avoid sharing the main-thread client.
    """
    if has_day_data_grib(t["day_dir"]):
        if t["zip_path"].exists() and not KEEP_ZIPS:
            safe_unlink(t["zip_path"])
        return True

    if t["zip_path"].exists() and not is_valid_zip(t["zip_path"]):
        safe_unlink(t["zip_path"])

    local_ds = DSClient(url=EWDS_URL, key=EWDS_TOKEN)

    for attempt in range(1, MAX_DOWNLOAD_RETRIES + 1):
        try:
            remote = local_ds.get_remote(request_id)
            remote.download(str(t["zip_path"]))

            if not is_valid_zip(t["zip_path"]):
                raise RuntimeError("downloaded file is not a valid zip")

            _extract_and_normalize(t)
            return True

        except Exception as e:
            safe_unlink(t["zip_path"])
            sleep_s = min(120, 10 * attempt) + random.uniform(0, 3)
            tlog(f"  {tag}: download attempt {attempt} failed: {e} (retry in {sleep_s:.0f}s)")
            time.sleep(sleep_s)

    return False


def submit_task(t: dict, state: dict) -> str:
    tag = t["tag"]
    rec = state["tasks"][tag]

    for attempt in range(1, MAX_SUBMIT_RETRIES + 1):
        try:
            remote = ds.submit(DATASET, t["request"])
            rec["request_id"]  = remote.request_id
            rec["status"]      = "submitted"
            rec["attempts"]    = rec.get("attempts", 0) + 1
            rec["submitted_at"] = time.time()
            rec["last_error"]  = None
            save_state(state)
            tlog(f"  submit {tag} → {remote.request_id}")
            return remote.request_id
        except Exception as e:
            if is_bad_request_400(e):
                tlog(f"  {tag}: 400 Bad Request — check request parameters: {e}")
                raise
            sleep_s = min(60, 5 * attempt) + random.uniform(0, 2)
            tlog(f"  submit failed {tag} attempt {attempt}: {e} (retry in {sleep_s:.0f}s)")
            time.sleep(sleep_s)

    raise RuntimeError(f"Submit permanently failed: {tag}")


def reconcile_job_deleted(t: dict, state: dict) -> str:
    tag = t["tag"]
    rec = state["tasks"][tag]

    if has_day_data_grib(t["day_dir"]):
        rec["status"]     = "done"
        rec["last_error"] = "job_deleted_but_data_present"
        save_state(state)
        tlog(f"  {tag}: job deleted but data.grib exists → done")
        return "done"

    if is_valid_zip(t["zip_path"]):
        try:
            _extract_and_normalize(t)
            rec["status"]     = "done"
            rec["last_error"] = "job_deleted_zip_recovered"
            save_state(state)
            tlog(f"  {tag}: job deleted but ZIP recovered → done")
            return "done"
        except Exception as e:
            safe_unlink(t["zip_path"])
            tlog(f"  {tag}: ZIP recovery failed: {e}")

    rec["status"]     = "pending"
    rec["request_id"] = None
    rec["last_error"] = "job_deleted_resubmit"
    save_state(state)
    tlog(f"  {tag}: job deleted and no data → will resubmit")
    return "resubmit"


# ── Main two-phase download loop ──────────────────────────────────────────────

def run_two_phase_download(tasks: list[dict], state: dict) -> None:
    """
    Interleaved Phase 1 (submit pending tasks up to MAX_INFLIGHT) and
    Phase 2 (poll inflight jobs; download + extract as they become ready).

    All 57 tasks across all events are treated as a flat list — no per-event
    outer loop. The loop terminates when every task is marked 'done'.
    """
    tag_to_task = {t["tag"]: t for t in tasks}
    total       = len(tasks)
    done_count  = sum(1 for t in tasks if state["tasks"][t["tag"]]["status"] == "done")

    # Re-register any jobs that were inflight at last crash
    inflight: dict[str, str] = {}
    for t in tasks:
        rec = state["tasks"][t["tag"]]
        rid = rec.get("request_id")
        if rid and rec.get("status") in ("submitted", "running", "ready", "downloading"):
            inflight[t["tag"]] = rid

    pbar = tqdm(total=total, initial=done_count, desc="Forecast days")

    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
        download_futures: dict[str, object] = {}
        ready_queue: list[str] = []

        def schedule_download(tag: str) -> bool:
            if tag in download_futures or len(download_futures) >= DOWNLOAD_WORKERS:
                return False
            t   = tag_to_task[tag]
            rec = state["tasks"][tag]
            rid = rec.get("request_id")
            if not rid:
                return False
            rec["status"] = "downloading"
            save_state(state)
            fut = executor.submit(download_day_worker, tag, rid, t)
            download_futures[tag] = fut
            tlog(f"  download queued {tag} ({len(download_futures)}/{DOWNLOAD_WORKERS} active)")
            return True

        while True:
            # 1) Harvest completed downloads
            finished = [k for k, v in download_futures.items() if v.done()]
            for tag in finished:
                fut = download_futures.pop(tag)
                rec = state["tasks"][tag]
                try:
                    ok = fut.result()
                except Exception as e:
                    tlog(f"  {tag}: worker exception: {e}")
                    ok = False

                if ok:
                    rec["status"] = "done"
                    save_state(state)
                    pbar.update(1)
                    tlog(f"  done  {tag}  ({mb(tag_to_task[tag]['day_dir'] / DAY_OUT_NAME):.1f} MB)")
                else:
                    rec["status"] = "ready"
                    save_state(state)
                    if tag not in ready_queue:
                        ready_queue.append(tag)
                    tlog(f"  {tag}: download failed → queued for retry")

            # 2) Progress + termination
            done_count = sum(1 for t in tasks if state["tasks"][t["tag"]]["status"] == "done")
            pbar.n = done_count
            pbar.refresh()
            pbar.set_postfix(inflight=len(inflight), dl=len(download_futures), q=len(ready_queue))

            if done_count >= total and not download_futures:
                break

            progressed = False

            # 3) Drain ready_queue into download pool
            while ready_queue and len(download_futures) < DOWNLOAD_WORKERS:
                tag = ready_queue.pop(0)
                if state["tasks"][tag]["status"] == "done":
                    continue
                if schedule_download(tag):
                    progressed = True

            # 4) Phase 1: submit pending tasks up to MAX_INFLIGHT
            for t in tasks:
                if len(inflight) >= MAX_INFLIGHT:
                    break
                tag = t["tag"]
                rec = state["tasks"][tag]
                st  = rec["status"]
                if (
                    st == "done"
                    or tag in inflight
                    or tag in download_futures
                    or tag in ready_queue
                    or st in ("submitted", "running", "ready", "downloading")
                ):
                    continue
                try:
                    rid = submit_task(t, state)
                    inflight[tag] = rid
                    progressed = True
                except Exception as e:
                    tlog(f"  {tag}: submit error — skipping this round: {e}")
                    rec["last_error"] = str(e)[:300]
                    save_state(state)

            # 5) Phase 2: poll inflight jobs
            for tag in list(inflight.keys()):
                rec = state["tasks"][tag]
                if rec["status"] == "done" or tag in download_futures:
                    inflight.pop(tag, None)
                    continue

                rid = inflight[tag]
                try:
                    remote = ds.get_remote(rid)
                except Exception as e:
                    if is_job_not_found_error(e):
                        rec["last_error"] = str(e)[:300]
                        save_state(state)
                        action = reconcile_job_deleted(tag_to_task[tag], state)
                        inflight.pop(tag, None)
                        progressed = True
                        if action == "done":
                            pbar.update(1)
                    else:
                        tlog(f"  poll failed {tag}: {e}")
                        rec["last_error"] = str(e)[:300]
                        save_state(state)
                    continue

                rec["last_poll"] = time.time()
                save_state(state)

                status = getattr(remote, "status", None)
                ready  = getattr(remote, "results_ready", False)

                if status in ("successful", "success") and ready:
                    rec["status"] = "ready"
                    save_state(state)
                    inflight.pop(tag, None)
                    if not schedule_download(tag):
                        if tag not in ready_queue:
                            ready_queue.append(tag)
                            tlog(f"  {tag}: queued (no download slot available)")
                    progressed = True

                elif status in ("failed", "dismissed", "deleted"):
                    tlog(f"  {tag}: status={status} → will resubmit")
                    rec["status"]     = "pending"
                    rec["request_id"] = None
                    save_state(state)
                    inflight.pop(tag, None)
                    progressed = True

                else:
                    rec["status"] = "running"
                    save_state(state)

            if not progressed:
                time.sleep(POLL_SECONDS + random.uniform(0, JITTER_SECONDS))

    pbar.close()


# ── Run ───────────────────────────────────────────────────────────────────────
run_two_phase_download(tasks, state)
print("All done.")

Forecast days:   0%|          | 0/56 [00:00<?, ?it/s]

  submit EV_MARCE2024_2024-10-24 → bfda3907-75a9-4814-af22-7e80dbae78be
  submit EV_MARCE2024_2024-10-25 → 30297e53-29b7-4202-93cf-a09d3d820ec7
  submit EV_MARCE2024_2024-10-26 → 11c0bb1c-003a-4777-8c0d-c4c9e808139a
  submit EV_MARCE2024_2024-10-27 → 26cc2004-9534-4c51-b033-57a2a8d218ae
  submit EV_MARCE2024_2024-10-28 → 29340209-decc-48a7-ac32-0f6a27540d2f
  submit EV_MARCE2024_2024-10-29 → eb3c1ff4-d419-42de-8378-5c7d5845fcb3
  submit EV_MARCE2024_2024-10-30 → cbb52fda-277b-488d-b025-e87b7235dd6f
  submit EV_MARCE2024_2024-10-31 → 8ac30716-dab5-4344-af08-5fe989526743
  submit EV_MARCE2024_2024-11-01 → b9af27fe-3b99-4d16-9ca1-90171d537d9a
  submit EV_MARCE2024_2024-11-02 → 79e7dcfc-056a-4edd-a0ba-dab2ad8eae64
  submit EV_MARCE2024_2024-11-03 → 2faa2487-d0f6-4fd9-a995-233e99ccca5b
  submit EV_MARCE2024_2024-11-04 → 0ef22b44-41e5-40dc-802f-250622cddaf2
  download queued EV_MARCE2024_2024-10-24 (1/4 active)


f8422f8fabe112deb31be87e3176151f.zip:   0%|          | 0.00/504M [00:00<?, ?B/s]

  submit EV_MARCE2024_2024-11-05 → 32d839b4-e059-4a61-9e27-c5180f495637
  done  EV_MARCE2024_2024-10-24  (964.0 MB)
  download queued EV_MARCE2024_2024-10-25 (1/4 active)


d430cb96401044ced20e7dd4d3f5715e.zip:   0%|          | 0.00/503M [00:00<?, ?B/s]

...

  done  EV_NEM2025_2025-12-01  (964.0 MB)
  download queued EV_NEM2025_2025-12-02 (1/4 active)


d5a7c4e0002741525ca7bde29c6fa012.zip:   0%|          | 0.00/395M [00:00<?, ?B/s]

  EV_NEM2025_2025-12-02: download attempt 1 failed: File size mismatch 414187520 bytes instead of 414243487 (retry in 12s)


d5a7c4e0002741525ca7bde29c6fa012.zip:   0%|          | 0.00/395M [00:00<?, ?B/s]

  done  EV_NEM2025_2025-12-02  (964.0 MB)
All done.


In [7]:
# Cell 7 — Verification

print("=" * 60)
print("Output verification")
print("=" * 60)

all_ok = True
for event_id, ev in EVENTS.items():
    peak    = date.fromisoformat(ev["peak_date"])
    start_d = peak - timedelta(days=WINDOW_PRE_DAYS)
    end_d   = peak + timedelta(days=WINDOW_POST_DAYS)
    n_expected = (end_d - start_d).days + 1

    missing = []
    d = start_d
    while d <= end_d:
        grib = RAW_ROOT / event_id / d.isoformat() / DAY_OUT_NAME
        if not is_grib_payload(grib):
            missing.append(d.isoformat())
        d += timedelta(days=1)

    n_ok = n_expected - len(missing)
    status_str = "OK" if not missing else f"MISSING {len(missing)}"
    print(f"  {event_id:20s} ({ev['label']:30s}):  {n_ok}/{n_expected}  [{status_str}]")
    for m in missing:
        print(f"      missing: {m}")
    if missing:
        all_ok = False

# State counter summary
statuses = [state["tasks"][t["tag"]]["status"] for t in tasks]
print()
print("State summary:", dict(Counter(statuses)))

# Clean up zips staging dirs if everything is done
if all_ok:
    for event_id in EVENTS:
        zips_dir = RAW_ROOT / event_id / "_staging" / "zips"
        safe_rmtree(zips_dir)
    print()
    print("All events complete. Temporary zips dirs cleaned.")
else:
    print()
    print("Some days are missing — re-run Cell 6 to resume.")

Output verification
  EV_MARCE2024         (TY Marce — Nov 2024           ):  19/19  [OK]
  EV_UWAN2025          (STY Uwan — Nov 2025           ):  19/19  [OK]
  EV_NEM2025           (NE Monsoon — Dec 2025         ):  19/19  [OK]

State summary: {'done': 56}

All events complete. Temporary zips dirs cleaned.
